# Pemodelan Struktural (Dimension Modeling)


## Inisialisasi Spark Session dan Setup Path


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Inisialisasi SparkSession dengan konfigurasi dynamic partition overwrite
spark = (
    SparkSession.builder
    .appName('Retail_Dimensional_Modeling')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic') # Kunci idempotensi untuk overwrite partisi secara dinamis
    .getOrCreate()
)

raw_path = '../data/raw/olist'
silver_path = '../data/silver'

print("SparkSession initialized successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/18 10:18:13 WARN Utils: Your hostname, lathief-laptop, resolves to a loopback address: 127.0.1.1; using 192.168.1.9 instead (on interface wlp0s20f3)
26/09/18 10:18:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lathief/coding/myproject/project4-aws/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/18 10:18:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialized successfully.


## 2. Membangun Dimensi Kalender (dim_date)


In [2]:
df_date_range = spark.sql(
    """
    select
        explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as calendar_date
    """
)

dim_date = (
    df_date_range
    .withColumn('date_key', F.date_format('calendar_date', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('year', F.year('calendar_date'))
    .withColumn('month', F.month('calendar_date'))
    .withColumn('month_name', F.date_format('calendar_date', 'MMMM'))
    .withColumn('day', F.dayofmonth('calendar_date'))
    .withColumn('day_of_week', F.dayofweek('calendar_date'))
    .withColumn('day_name', F.date_format('calendar_date', 'EEEE'))
    .withColumn('quarter', F.quarter('calendar_date'))
    .withColumn('is_weekend', F.when(F.col('day_of_week').isin(1,7), True).otherwise(False))
)

dim_date.printSchema()
dim_date.show(5, truncate=False)

root
 |-- calendar_date: date (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- day: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- day_name: string (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- is_weekend: boolean (nullable = false)

+-------------+--------+----+-----+----------+---+-----------+--------+-------+----------+
|calendar_date|date_key|year|month|month_name|day|day_of_week|day_name|quarter|is_weekend|
+-------------+--------+----+-----+----------+---+-----------+--------+-------+----------+
|2016-01-01   |20160101|2016|1    |January   |1  |6          |Friday  |1      |false     |
|2016-01-02   |20160102|2016|1    |January   |2  |7          |Saturday|1      |true      |
|2016-01-03   |20160103|2016|1    |January   |3  |1          |Sunday  |1      |true      |
|2016-01-04   |20160104|2016|1    |

In [3]:
# Tulis ke silver layer (Format parquet, overwrite mode)
dim_date.write.mode('overwrite').parquet(f'{silver_path}/dim_date')
print('dim_date successfully saved to silver layer.')

dim_date successfully saved to silver layer.


## 3. Membangun Dimensi Produk (dim_product) - scd-type 1


In [4]:
df_products_raw = spark.read.csv(f'{raw_path}/olist_products_dataset.csv', header=True, inferSchema=True)
df_categories_raw = spark.read.csv(f'{raw_path}/product_category_name_translation.csv', header=True, inferSchema=True)

# 1. Enrichment translasi kategori & sanitasi missing values
dim_product = (
    df_products_raw
    .join(df_categories_raw, on='product_category_name', how='left')
    .select(
        F.col('product_id'),
        # Imputasi: jika null, beri label 'Unknown' agar join downstream tidak pecah
        F.coalesce(F.col('product_category_name_english'), F.lit('unknown')).alias('category_name'),
        F.coalesce(F.col('product_weight_g'), F.lit(0)).alias('weight_g'),
        F.coalesce(F.col('product_length_cm'), F.lit(0)).alias('length_cm'),
        F.coalesce(F.col('product_height_cm'), F.lit(0)).alias('height_cm'),
        F.coalesce(F.col('product_width_cm'), F.lit(0)).alias('width_cm'),
        # Timestamp metadata audit (kapan record ini diproses ke Silver)
        F.current_timestamp().alias('updated_at')
    )
    .dropDuplicates(['product_id'])  # Menjamin uniqueness Axiom: product_id adalah primary key
)

dim_product.printSchema()
dim_product.show(5, truncate=False)

root
 |-- product_id: string (nullable = true)
 |-- category_name: string (nullable = false)
 |-- weight_g: integer (nullable = false)
 |-- length_cm: integer (nullable = false)
 |-- height_cm: integer (nullable = false)
 |-- width_cm: integer (nullable = false)
 |-- updated_at: timestamp (nullable = false)

+--------------------------------+--------------+--------+---------+---------+--------+--------------------------+
|product_id                      |category_name |weight_g|length_cm|height_cm|width_cm|updated_at                |
+--------------------------------+--------------+--------+---------+---------+--------+--------------------------+
|00066f42aeeb9f3007548bb9d3f33c38|perfumery     |300     |20       |16       |16      |2026-09-18 03:18:20.176852|
|00088930e925c41fd95ebfe695fd2655|auto          |1225    |55       |10       |26      |2026-09-18 03:18:20.176852|
|0011c512eb256aa0dbbb544d8dffcf6e|auto          |100     |16       |15       |16      |2026-09-18 03:18:20.176852|


In [5]:
dim_product.write.mode('overwrite').parquet(f'{silver_path}/dim_product')
print('dim_product successfully saved to silver layer.')

dim_product successfully saved to silver layer.


## 4. Membangun dim_customer scd-type2


In [6]:
from pyspark.sql.window import Window

df_customers_raw = spark.read.csv(f'{raw_path}/olist_customers_dataset.csv', header=True, inferSchema=True)
df_orders_raw = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)

# 1. Hubungkan customer dengan purchase timestamp dari order untuk tahu kapan profil ini aktif
df_cust_orders = (
    df_customers_raw
    .join(df_orders_raw.select('order_id', 'customer_id', 'order_purchase_timestamp'), on='customer_id', how='inner')
    .withColumn('purchase_date', F.to_date(F.to_timestamp('order_purchase_timestamp')))
    .select(
        'customer_unique_id',
        F.col('customer_zip_code_prefix').alias('zip_code'),
        F.col('customer_city').alias('city'),
        F.col('customer_state').alias('state'),
        'purchase_date'
    )
)

# 2. Ambil tanggal pertama kali kombinasi (customer + lokasi) muncul (effective start date)
df_cust_history = (
    df_cust_orders
    .groupBy('customer_unique_id', 'zip_code', 'city', 'state')
    .agg(F.min('purchase_date').alias('start_date'))
)

# 3. Urutkan riwayat per customer_unique_id menggunakan window function untuk menentukan end_date & is_current
window_spec = Window.partitionBy('customer_unique_id').orderBy('start_date')

dim_customer_scd2 = (
    df_cust_history
    # end_date adalah start_date dari versi berikutnya (lead), atau '9999-12-31' jika versi terbaru
    .withColumn('next_start_date', F.lead('start_date').over(window_spec))
    .withColumn('end_date', F.coalesce(F.date_sub(F.col('next_start_date'), 1), F.to_date(F.lit('9999-12-31'))))  # Jika tidak ada next_start_date, berarti ini adalah versi terakhir
    # is_current bernilai True hanya jika record tidak memiliki versi penerus (next_start_date is null)
    .withColumn('is_current', F.col('next_start_date').isNull())
    # Generate deterministic surrogate key untuk dim_customer_scd2 pakai MD5 hash
    .withColumn('customer_sk', F.md5(F.concat_ws('||', 'customer_unique_id', 'city', 'state', 'start_date')))
    .select(
        'customer_sk',
        'customer_unique_id',
        'zip_code',
        'city',
        'state',
        'start_date',
        'end_date',
        'is_current'
    )
)


dim_customer_scd2.printSchema()
dim_customer_scd2.show(5, truncate=False)

root
 |-- customer_sk: string (nullable = false)
 |-- customer_unique_id: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = false)

+--------------------------------+--------------------------------+--------+--------------+-----+----------+----------+----------+
|customer_sk                     |customer_unique_id              |zip_code|city          |state|start_date|end_date  |is_current|
+--------------------------------+--------------------------------+--------+--------------+-----+----------+----------+----------+
|976363f92dd06b4f94d6566de7b3be4f|0006fdc98a402fceb4eb0ee528f6a8d4|29400   |mimoso do sul |ES   |2017-07-18|9999-12-31|true      |
|a80f9b868fe0d7a5ce127700f49b3037|00090324bbad0e9342388303bb71ba0a|13054   |campinas      |SP   |2018-03-24|9999-12-31|true      |
|6eb24d81

## 5. Verifikasi Riwayat Customer yang Pindah Lokasi


In [7]:
# Ambil contoh customer yang di-EDA terbukti pindah kota
sample_customer = 'd44ccec15f5f86d14d6a2cfa67da1975'

print(f'Riwayat SCD2 untuk customer: {sample_customer}')
dim_customer_scd2.filter(F.col('customer_unique_id') == sample_customer).show(truncate=False)

# Simpan ke silver layer
dim_customer_scd2.write.mode('overwrite').parquet(f'{silver_path}/dim_customer')
print('dim_customer (SCD Type 2) successfully saved to silver layer.')

Riwayat SCD2 untuk customer: d44ccec15f5f86d14d6a2cfa67da1975
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+
|customer_sk                     |customer_unique_id              |zip_code|city      |state|start_date|end_date  |is_current|
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+
|b56262969fb77822453670af84965964|d44ccec15f5f86d14d6a2cfa67da1975|3533    |sao paulo |SP   |2017-05-30|2017-09-12|false     |
|9731c4c7c012add288fc23e284402141|d44ccec15f5f86d14d6a2cfa67da1975|88371   |navegantes|SC   |2017-09-13|2017-11-09|false     |
|eb9476e48511f61c8392d4bd547b5fc0|d44ccec15f5f86d14d6a2cfa67da1975|62800   |aracati   |CE   |2017-11-10|9999-12-31|true      |
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+

dim_customer (SCD Type 2) successfully saved to 

## 6. Membangun fact_orders dengan Point-in-Time SCD2 Lookup


In [8]:
df_items_raw = spark.read.csv(f'{raw_path}/olist_order_items_dataset.csv', header=True, inferSchema=True)
df_order_raw = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)
df_customers_raw = spark.read.csv(f'{raw_path}/olist_customers_dataset.csv', header=True, inferSchema=True)

# Baca dim_customer yang sudah tersimpan di Silver
dim_customer = spark.read.parquet(f'{silver_path}/dim_customer')

# 1. Siapkan transaksi dasar (orders + items + customer_unique_id)
df_transaksi = (
    df_items_raw
    .join(df_orders_raw, on='order_id', how='inner')
    .join(df_customers_raw.select('customer_id', 'customer_unique_id'), on='customer_id', how='inner')
    .withColumn('purchase_ts', F.to_timestamp('order_purchase_timestamp'))
    .withColumn('purchase_date', F.to_date('purchase_ts'))
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast(IntegerType()))
)

# 2. Poin-in-Time Join ke dim_customer (Range Join: purchase_date BETWEEN start_date AND end_date)
fact_orders = (
    df_transaksi
    .join(dim_customer, on=(
        (df_transaksi.customer_unique_id == dim_customer.customer_unique_id)
        & (df_transaksi.purchase_date >= dim_customer.start_date)
        & (df_transaksi.purchase_date <= dim_customer.end_date)
    ), how='inner')
    .select(
        # Degenerate Dimension / Business Key
        df_transaksi['order_id'],
        df_transaksi['order_item_id'],

        # Foreign Key ke Dimensi
        df_transaksi['date_key'],
        df_transaksi['product_id'],
        dim_customer['customer_sk'],

        # Transaction Status
        df_transaksi['order_status'],

        # Additive Fact Measures (Nilai Numerik yang Sah di-SUM)
        df_transaksi['price'].alias('item_price'),
        df_transaksi['freight_value'],
        (df_transaksi['price'] + df_transaksi['freight_value']).alias('total_item_value'),

        # Kolom Partisi Waktu (Untuk Storage & Idempotensi)
        df_transaksi['purchase_date']
    )
)

print(f'Total baris Fact Orders: {fact_orders.count()}')
fact_orders.printSchema()
fact_orders.show(5, truncate=False)

Total baris Fact Orders: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- item_price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- purchase_date: date (nullable = true)

+--------------------------------+-------------+--------+--------------------------------+--------------------------------+------------+----------+-------------+----------------+-------------+
|order_id                        |order_item_id|date_key|product_id                      |customer_sk                     |order_status|item_price|freight_value|total_item_value|purchase_date|
+--------------------------------+-------------+--------+--------------------------------+--------------------------------+------------+------

## 7. Menulis ke Silver dengan Partisi Harian


In [9]:
# Tulis ke Silver Layer dengan partisi harian
(
    fact_orders
    .write
    .mode('overwrite')
    .partitionBy('purchase_date')
    .parquet(f'{silver_path}/fact_orders')
)

print('fact_orders successfully saved to silver layer with daily partitioning.')

fact_orders successfully saved to silver layer with daily partitioning.


## 8. The Idempotency Test


In [10]:
# Pilih satu tanggal spesifik untuk uji rerun
test_date = '2017-10-02'

# 1. Hitung total baris awal di tanggal tersebut
initial_count = spark.read.parquet(f'{silver_path}/fact_orders').filter(F.col('purchase_date') == test_date).count()
print(f'Jumlah baris awal untuk {test_date}: {initial_count}')

# 2. Simulasikan Airflow mmenjalankan ulang batch untuk tanggal 2017-10-02 saja
single_day_batch = fact_orders.filter(F.col('purchase_date') == test_date)

# Tulis ulang dengan mode overwrite (karena spark.sql.sources.partitionOverWriteMode = dynamic, Hanya partisi 2017-10-02 yang disentuh)
(
    single_day_batch
    .write
    .mode('overwrite')
    .partitionBy('purchase_date')
    .parquet(f'{silver_path}/fact_orders')
)

# 3. hitung ulang total baris setelah rerun
post_rerun_count = spark.read.parquet(f'{silver_path}/fact_orders').filter(F.col('purchase_date') == test_date).count()
print(f'Jumlah baris setelah rerun untuk {test_date}: {post_rerun_count}')

# 4. Assert Idempotensi: initial_count HARUS SAMA DENGAN post_rerun_count
assert initial_count == post_rerun_count, 'Gagal: Terjadi duplikasi data!'
print('Idempotency Assertion Passed: Tidak ada duplikasi data saat pipeline dijalankan ulang.')


Jumlah baris awal untuk 2017-10-02: 166


Jumlah baris setelah rerun untuk 2017-10-02: 166
Idempotency Assertion Passed: Tidak ada duplikasi data saat pipeline dijalankan ulang.


## 9. Membangun dim_geolocation (Reference Dim)

Sumber: `olist_geolocation_dataset.csv`. 1 zip prefix punya banyak titik koordinat -> `GROUP BY zip_code_prefix` dan ambil rata-rata lat/lng.


In [11]:
df_geo_raw = spark.read.csv(f'{raw_path}/olist_geolocation_dataset.csv', header=True, inferSchema=True)

dim_geolocation = (
    df_geo_raw
    .withColumnRenamed('geolocation_zip_code_prefix', 'zip_code_prefix')
    .groupBy('zip_code_prefix')
    .agg(
        F.mean('geolocation_lat').alias('lat'),
        F.mean('geolocation_lng').alias('lng'),
        F.first('geolocation_city').alias('city'),
        F.first('geolocation_state').alias('state'),
    )
    .select(
        'zip_code_prefix',
        F.round('lat', 6).alias('lat'),
        F.round('lng', 6).alias('lng'),
        'city',
        'state',
    )
)

# Broadcast join untuk enrichment customer & seller nanti
dim_geolocation.write.mode('overwrite').parquet(f'{silver_path}/dim_geolocation')
print(f'dim_geolocation saved. Rows: {dim_geolocation.count()}')

# Validasi
dim_geolocation.show(5, truncate=False)


dim_geolocation saved. Rows: 19015
+---------------+----------+----------+---------+-----+
|zip_code_prefix|lat       |lng       |city     |state|
+---------------+----------+----------+---------+-----+
|1001           |-23.55019 |-46.634024|sao paulo|SP   |
|1002           |-23.548146|-46.634979|sao paulo|SP   |
|1003           |-23.548994|-46.635731|sao paulo|SP   |
|1004           |-23.549799|-46.634757|sao paulo|SP   |
|1005           |-23.549456|-46.636733|sao paulo|SP   |
+---------------+----------+----------+---------+-----+
only showing top 5 rows


## 10. Membangun dim_seller (SCD Type 1)

Enrich seller dengan lat/lng dari `dim_geolocation` (broadcast join). Ubah master data -> current state only.


In [12]:
df_sellers_raw = spark.read.csv(f'{raw_path}/olist_sellers_dataset.csv', header=True, inferSchema=True)

dim_seller = (
    df_sellers_raw
    .join(
        dim_geolocation.hint('broadcast'),
        on=df_sellers_raw['seller_zip_code_prefix'] == dim_geolocation['zip_code_prefix'],
        how='left'
    )
    .select(
        'seller_id',
        F.col('seller_zip_code_prefix').alias('seller_zip_code_prefix'),
        F.col('seller_city').alias('city'),
        F.col('seller_state').alias('state'),
        'lat',
        'lng',
        F.current_timestamp().alias('updated_at'),
    )
    .dropDuplicates(['seller_id'])
)

dim_seller.write.mode('overwrite').parquet(f'{silver_path}/dim_seller')
print(f'dim_seller saved. Rows: {dim_seller.count()}')

# Validasi
dim_seller.show(5, truncate=False)


dim_seller saved. Rows: 3095
+--------------------------------+----------------------+-----------+-----+----------+----------+-------------------------+
|seller_id                       |seller_zip_code_prefix|city       |state|lat       |lng       |updated_at               |
+--------------------------------+----------------------+-----------+-----+----------+----------+-------------------------+
|0015a82c2db000af6aaaf3ae2ecb0532|9080                  |santo andre|SP   |-23.640444|-46.541742|2026-09-18 03:18:50.72954|
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES   |-20.278513|-40.411675|2026-09-18 03:18:50.72954|
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ   |-22.872355|-43.027433|2026-09-18 03:18:50.72954|
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP   |-20.528759|-47.41111 |2026-09-18 03:18:50.72954|
|003554e2dce176b5555353e4f3555ac8|74565                 |goiania    |GO   |-16.640574|-49.276483|2026-0

## 11. Membangun fact_order_payments (Append-only Fact)

Grain: `order_id + payment_sequential`. Partisi `purchase_date` (join orders) untuk sinkron idempotensi harian & backfill.


In [13]:
df_payments_raw = spark.read.csv(f'{raw_path}/olist_order_payments_dataset.csv', header=True, inferSchema=True)
df_orders_raw = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)

fact_order_payments = (
    df_payments_raw
    .join(df_orders_raw.select('order_id', 'order_purchase_timestamp'), on='order_id', how='inner')
    .withColumn('purchase_date', F.to_date(F.to_timestamp('order_purchase_timestamp')))
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast('int'))
    .filter(F.col('purchase_date').isNotNull())
    .select(
        'order_id',
        'payment_sequential',
        'payment_type',
        'payment_installments',
        'payment_value',
        'purchase_date',
        'date_key',
    )
)

fact_order_payments.write.mode('overwrite').partitionBy('purchase_date').parquet(f'{silver_path}/fact_order_payments')
print(f'fact_order_payments saved. Rows: {fact_order_payments.count()}')

# Validasi
fact_order_payments.show(5, truncate=False)


fact_order_payments saved. Rows: 103886
+--------------------------------+------------------+------------+--------------------+-------------+-------------+--------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|purchase_date|date_key|
+--------------------------------+------------------+------------+--------------------+-------------+-------------+--------+
|e481f51cbdc54678b7cc49136f2d6af7|2                 |voucher     |1                   |18.59        |2017-10-02   |20171002|
|e481f51cbdc54678b7cc49136f2d6af7|3                 |voucher     |1                   |2.0          |2017-10-02   |20171002|
|e481f51cbdc54678b7cc49136f2d6af7|1                 |credit_card |1                   |18.12        |2017-10-02   |20171002|
|53cdb2fc8bc7dce0b6741e2150273451|1                 |boleto      |1                   |141.46       |2018-07-24   |20180724|
|47770eb9100c2d0c44946d9cf07ec65d|1                 |credit_card |3                  

## 12. Membangun fact_order_reviews (Append-only Fact)

Grain: `review_id`. Partisi `purchase_date` dari order (point-in-time). Measure utama: `review_score`.


In [15]:
# Data review punya beberapa baris dengan komentar multi-line / tanggal tak standar.
# Jadi kita baca dengan mode permissive dan hanya konversi timestamp bila format memang valid.
df_reviews_raw = (
    spark.read
    .option('multiLine', 'true')
    .option('mode', 'PERMISSIVE')
    .option('quote', '"')
    .option('escape', '"')
    .csv(f'{raw_path}/olist_order_reviews_dataset.csv', header=True, inferSchema=False)
)

# Normalisasi kolom agar urutan tetap konsisten
review_columns = [
    'review_id',
    'order_id',
    'review_score',
    'review_comment_title',
    'review_comment_message',
    'review_creation_date',
    'review_answer_timestamp',
]
df_reviews_raw = df_reviews_raw.toDF(*review_columns)

fact_order_reviews = (
    df_reviews_raw
    .join(df_orders_raw.select('order_id', 'order_purchase_timestamp'), on='order_id', how='inner')
    .withColumn('purchase_date', F.to_date(F.to_timestamp('order_purchase_timestamp')))
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast('int'))
    .withColumn(
        'review_creation_date',
        F.when(
            F.col('review_creation_date').rlike(r'^\d{4}-\d{2}-\d{2}.*$'),
            F.to_timestamp('review_creation_date', 'yyyy-MM-dd HH:mm:ss')
        ).otherwise(None)
    )
    .withColumn('review_creation_date', F.to_date('review_creation_date'))
    .filter(F.col('purchase_date').isNotNull())
    .select(
        'review_id',
        'order_id',
        F.col('review_score').cast('int').alias('review_score'),
        'review_comment_title',
        'review_comment_message',
        'review_creation_date',
        'purchase_date',
        'date_key',
    )
    .dropDuplicates(['review_id'])
)

fact_order_reviews.write.mode('overwrite').partitionBy('purchase_date').parquet(f'{silver_path}/fact_order_reviews')
print(f'fact_order_reviews saved. Rows: {fact_order_reviews.count()}')

# Validasi
fact_order_reviews.show(5, truncate=False)


fact_order_reviews saved. Rows: 98410
+--------------------------------+--------------------------------+------------+--------------------+--------------------------------------------------------------------------------------------------+--------------------+-------------+--------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                            |review_creation_date|purchase_date|date_key|
+--------------------------------+--------------------------------+------------+--------------------+--------------------------------------------------------------------------------------------------+--------------------+-------------+--------+
|000932bcd6959ac688b310abead1acb8|c72add79a1bef8c55628b377b01eb02d|4           |NULL                |NULL                                                                                              |2017-04-06          |2017-0

## 13. Update fact_orders: tambah FK seller_sk

Rejoin `fact_orders` ke `dim_seller` via `seller_id` dari `order_items` untuk melengkapi Star Schema.


In [16]:
# Siapkan mapping seller dari order_items
df_items_seller = df_items_raw.select('order_id', 'order_item_id', 'seller_id')

# Rejoin fact_orders ke seller_id (item grain) lalu ke dim_seller untuk FK seller_sk
fact_orders_full = (
    fact_orders
    .join(df_items_seller, on=['order_id', 'order_item_id'], how='inner')
    .join(dim_seller.select('seller_id', F.col('seller_id').alias('seller_sk')), on='seller_id', how='left')
    .drop('seller_id')
)

# Tulis ulang dengan partisi dinamis (idempotent)
fact_orders_full.write.mode('overwrite').partitionBy('purchase_date').parquet(f'{silver_path}/fact_orders')
print(f'fact_orders updated with seller_sk. Rows: {fact_orders_full.count()}')

# Validasi
fact_orders_full.select('order_id', 'order_item_id', 'seller_sk', 'purchase_date').show(5, truncate=False)


fact_orders updated with seller_sk. Rows: 112650
+--------------------------------+-------------+--------------------------------+-------------+
|order_id                        |order_item_id|seller_sk                       |purchase_date|
+--------------------------------+-------------+--------------------------------+-------------+
|e481f51cbdc54678b7cc49136f2d6af7|1            |3504c0cb71d7fa48d967e0e4c94d59d9|2017-10-02   |
|53cdb2fc8bc7dce0b6741e2150273451|1            |289cdb325fb7e7f891c38608bf9e0962|2018-07-24   |
|47770eb9100c2d0c44946d9cf07ec65d|1            |4869f7a5dfa277a7dca6462dcf3b52b2|2018-08-08   |
|949d5b44dbf5de918fe9c16f97b45f8a|1            |66922902710d126a0e7d26b0e3805106|2017-11-18   |
|ad21c59c0840e6cb83a9ceb5573f8159|1            |2c9e548be18521d1c43cde1c582c6de8|2018-02-13   |
+--------------------------------+-------------+--------------------------------+-------------+
only showing top 5 rows
